# ICDAR Mini - Multilingual Text Recognition Benchmark Analysis

## Executive Summary

**Dataset:** ICDAR Mini - Multilingual Text Recognition  
**Total Samples:** 500 multilingual text images per phase  
**Task:** Multilingual optical character recognition (OCR)  
**Evaluation Metrics:** CER (Character Error Rate), WER (Word Error Rate), Cosine Similarity

## Benchmark Structure

### Phase Pa: OCR Baseline (Specialized OCR Models)
- **Models:** Azure Document Intelligence, Mistral Document AI
- **Approach:** Pure OCR on document images
- **Purpose:** Establish baseline OCR performance for multilingual text recognition

### Phase Pb: VLM Baseline (Generic Prompting)
- **Models:** GPT-5-nano, GPT-5-mini
- **Prompt:** Generic text extraction (no language-specific context)
- **Purpose:** Evaluate general-purpose VLM capabilities for multilingual OCR

### Phase Pc: VLM with Task-Aware Prompting
- **Models:** GPT-5-nano, GPT-5-mini
- **Prompt:** Multilingual-aware instructions
- **Purpose:** Evaluate impact of language-aware prompting on recognition accuracy

## Key Metrics

- **CER (Character Error Rate):** Edit distance at character level (lower is better, 0.0 = perfect)
- **WER (Word Error Rate):** Edit distance at word level (lower is better, 0.0 = perfect)
- **Cosine Similarity:** Semantic similarity using embeddings (higher is better, 1.0 = identical)
- **Ground Truth in Prediction:** Presence of ground truth text in model output (higher is better)

## Analysis Focus Areas

1. **OCR vs VLM Performance:** How do specialized OCR models compare to vision language models for multilingual text?
2. **Language-Specific Patterns:** Do certain models perform better on specific languages?
3. **Prompting Impact:** How much does multilingual-aware prompting improve VLM performance?
4. **Error Analysis:** Which samples are most challenging for the models?
5. **Efficiency Tradeoffs:** Speed vs accuracy analysis across models

## To Run This Analysis

1. Ensure consolidated data exists in `../../2_clean/ICDAR_mini/`
2. This notebook will load Pa, Pb, Pc results and generate:
   - Character and word error rate calculations
   - Semantic similarity analysis
   - Model comparison visualizations
   - Sample-level best/worst analysis
   - Inference time benchmarks
   - Language-specific performance breakdown

## 1. Imports and Metadata

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import sys
from typing import List, Dict, Optional, Tuple
import warnings
warnings.filterwarnings('ignore')

# Progress bar for long operations
from tqdm.notebook import tqdm

# Add ocr_vs_vlm to path for imports
# Get the notebook directory and add parent paths
notebook_dir = Path.cwd()
notebooks_dir = notebook_dir.parent  # 4_notebooks/
ocr_vs_vlm_base = notebooks_dir.parent.parent  # research-playground/

sys.path.insert(0, str(ocr_vs_vlm_base))
sys.path.insert(0, str(notebooks_dir))

from ocr_vs_vlm.metrics.evaluation_metrics import (
    calculate_cer,
    calculate_wer,
    compute_cosine_similarity,
    compute_anls,
    compute_exact_match,
    compute_ground_truth_in_prediction
)

# Import embedding cache manager for efficient cosine similarity computation
from ocr_vs_vlm.metrics.embedding_cache import EmbeddingCacheManager, save_embeddings_for_phase

# Import model ordering configuration for consistent display
from utils.model_order import MODEL_ORDER, sort_models, get_model_display_name

# Visualization settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
sns.set_palette("husl")
%matplotlib inline

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 1000)

print("✓ Imports loaded successfully")
print(f"✓ Model display order: {', '.join(MODEL_ORDER)}")
print(f"✓ Working directory: {notebook_dir}")
print(f"✓ Utils path added: {notebooks_dir}")


ModuleNotFoundError: No module named 'utils.model_order'

In [ ]:
print("test")

### Load Phase Data

In [ ]:
# Data paths
data_dir = Path("../../2_clean/ICDAR_mini")

# Load phase files
df_pa = pd.read_csv(data_dir / "Pa.csv")
df_pb = pd.read_csv(data_dir / "Pb.csv")
df_pc = pd.read_csv(data_dir / "Pc.csv")

print(f"Phase A (OCR baseline): {len(df_pa)} samples")
print(f"Phase B (VLM baseline): {len(df_pb)} samples")
print(f"Phase C (VLM + context): {len(df_pc)} samples")

# Display column names for reference
print("\nPhase A columns:", df_pa.columns.tolist())
print("Phase B columns:", df_pb.columns.tolist())
print("Phase C columns:", df_pc.columns.tolist())

In [ ]:
# Dataset configuration
DATASET_NAME = "ICDAR_mini"

# Initialize embedding cache manager
# This will load any previously computed embeddings from 3_embeddings/
EMBEDDINGS_DIR = Path("../../3_embeddings")
embedding_manager = EmbeddingCacheManager(DATASET_NAME, EMBEDDINGS_DIR)

print(f"\n📁 Dataset: {DATASET_NAME}")
print(f"📂 Embeddings directory: {EMBEDDINGS_DIR.resolve()}")
if embedding_manager.cache:
    print(f"✅ Loaded cached embeddings for phases: {', '.join(embedding_manager.cache.keys())}")
else:
    print("⚠️ No cached embeddings found - will compute on first run and save for future use")

## 2. Dataset Explorer

### Language Distribution

In [ ]:
# Language breakdown
if 'language' in df_pa.columns:
    language_counts = df_pa['language'].value_counts().sort_index()
    print("Language Distribution:")
    print(language_counts)
    print(f"\nTotal languages: {len(language_counts)}")
    
    # Visualization
    fig, ax = plt.subplots(figsize=(12, 6))
    language_counts.plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title('Sample Distribution by Language', fontsize=14, fontweight='bold')
    ax.set_xlabel('Language', fontsize=12)
    ax.set_ylabel('Number of Samples', fontsize=12)
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("Warning: 'language' column not found in dataset")

### Basic Statistics

In [ ]:
# Model definitions
models_pa = ['azure_intelligence', 'donut', 'mistral_document_ai']
models_pb = ['gpt-5-mini', 'gpt-5-nano']
models_pc = ['gpt-5-mini', 'gpt-5-nano']

all_models = models_pa + models_pb + models_pc

print("Phase A (OCR models):", models_pa)
print("Phase B (VLM baseline):", models_pb)
print("Phase C (VLM + context):", models_pc)

# Ground truth statistics
if 'ground_truth' in df_pa.columns:
    gt_lengths = df_pa['ground_truth'].str.len()
    print("\nGround Truth Text Statistics:")
    print(f"  Mean length: {gt_lengths.mean():.1f} characters")
    print(f"  Median length: {gt_lengths.median():.1f} characters")
    print(f"  Min length: {gt_lengths.min()} characters")
    print(f"  Max length: {gt_lengths.max()} characters")

### Sample Predictions

In [ ]:
# Display 10 random predictions for 3 selected models
selected_models = ['azure_intelligence', 'gpt-5-mini', 'gpt-5-nano']
sample_indices = np.random.choice(len(df_pa), size=min(10, len(df_pa)), replace=False)

for idx in sample_indices:
    print("=" * 100)
    print(f"Sample {idx}")
    
    # Get language if available
    if 'language' in df_pa.columns:
        print(f"Language: {df_pa.iloc[idx]['language']}")
    
    # Ground truth
    if 'ground_truth' in df_pa.columns:
        gt = df_pa.iloc[idx]['ground_truth']
        print(f"\nGround Truth: {gt[:200]}{'...' if len(gt) > 200 else ''}")
    
    # Predictions from selected models
    print("\nPredictions:")
    
    # Azure Intelligence (from Pa)
    if 'azure_intelligence' in selected_models and 'prediction_azure_intelligence' in df_pa.columns:
        pred = df_pa.iloc[idx]['prediction_azure_intelligence']
        print(f"  [azure_intelligence]: {pred[:200]}{'...' if len(str(pred)) > 200 else ''}")
    
    # GPT-5-mini (from Pb)
    if 'gpt-5-mini' in selected_models and 'prediction_gpt-5-mini' in df_pb.columns:
        pred = df_pb.iloc[idx]['prediction_gpt-5-mini']
        print(f"  [gpt-5-mini]: {pred[:200]}{'...' if len(str(pred)) > 200 else ''}")
    
    # GPT-5-nano (from Pc)
    if 'gpt-5-nano' in selected_models and 'prediction_gpt-5-nano' in df_pc.columns:
        pred = df_pc.iloc[idx]['prediction_gpt-5-nano']
        print(f"  [gpt-5-nano]: {pred[:200]}{'...' if len(str(pred)) > 200 else ''}")
    
    print()

print("=" * 100)

## 3. Metrics Evaluation

### Calculate CER, WER, Cosine Similarity for All Models

In [ ]:
# Helper function to calculate metrics for a single prediction (using embedding cache)
def calculate_sample_metrics(
    ground_truth: str, 
    prediction: str,
    phase: str,
    sample_id: str,
    model: str,
    emb_manager: EmbeddingCacheManager
) -> Dict[str, float]:
    """Calculate all metrics for a single sample with cached embeddings."""
    if pd.isna(prediction) or prediction == "":
        return {
            'cer': 1.0,
            'wer': 1.0,
            'cosine_similarity': 0.0,
            'ground_truth_in_prediction': 0.0,
        }
    
    # Use embedding manager for cosine similarity (with caching)
    cosine_sim = emb_manager.compute_cosine_similarity(
        phase=phase,
        ground_truth=ground_truth,
        prediction=str(prediction),
        sample_id=sample_id,
        model=model
    )
    
    return {
        'cer': calculate_cer(ground_truth, prediction),
        'wer': calculate_wer(ground_truth, prediction),
        'cosine_similarity': cosine_sim,
        'ground_truth_in_prediction': compute_ground_truth_in_prediction(str(prediction), [ground_truth]),
    }

In [ ]:
# Calculate metrics for all phases and models
# Use phase dictionaries for unified iteration
phase_dfs = {'Pa': df_pa, 'Pb': df_pb, 'Pc': df_pc}
all_results = []

for phase, df in phase_dfs.items():
    print(f"\n📊 Calculating metrics for {phase}...")
    
    # Get all prediction columns
    pred_cols = [col for col in df.columns if col.startswith('prediction_')]
    
    for pred_col in pred_cols:
        model = pred_col.replace('prediction_', '')
        print(f"   Processing model: {model}")
        
        # Calculate metrics for each sample with progress bar
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"   {model}", leave=False):
            metrics = calculate_sample_metrics(
                ground_truth=row['ground_truth'],
                prediction=row[pred_col],
                phase=phase,
                sample_id=row['sample_id'],
                model=model,
                emb_manager=embedding_manager
            )
            
            all_results.append({
                'sample_id': row['sample_id'],
                'phase': phase,
                'model': model,
                'language': row.get('language', 'unknown'),
                'cer': metrics['cer'],
                'wer': metrics['wer'],
                'cosine_similarity': metrics['cosine_similarity'],
                'ground_truth_in_prediction': metrics['ground_truth_in_prediction'],
            })

# Create results DataFrame
df_results = pd.DataFrame(all_results)

print(f"\n✓ Calculated metrics for {len(df_results)} predictions")
print(f"  Models: {df_results['model'].nunique()}")
print(f"  Phases: {df_results['phase'].nunique()}")
print(f"  Languages: {df_results['language'].nunique()}")

# Save embeddings incrementally
for phase in phase_dfs.keys():
    if phase in embedding_manager.modified_phases:
        print(f"   💾 Saving embeddings for {phase}...")
        saved_file = save_embeddings_for_phase(
            dataset_name=DATASET_NAME,
            phase=phase,
            embeddings_dict=embedding_manager.cache.get(phase, {}),
            embeddings_base_dir=EMBEDDINGS_DIR
        )
        print(f"   ✅ Saved: {saved_file.name}")

print("\n✅ Metrics calculation complete!")

### Aggregate Metrics by Model

In [ ]:
# Aggregate by model and phase
df_agg = df_results.groupby(['phase', 'model']).agg({
    'cer': ['mean', 'std', 'median'],
    'wer': ['mean', 'std', 'median'],
    'cosine_similarity': ['mean', 'std', 'median'],
    'sample_id': 'count'
}).round(4)

df_agg.columns = ['_'.join(col).strip() for col in df_agg.columns.values]
df_agg = df_agg.rename(columns={'sample_id_count': 'n_samples'})
df_agg = df_agg.reset_index()

print("Aggregated Metrics by Model and Phase:")
display(df_agg.sort_values(['phase', 'cer_mean']))

# Overall ranking (lower CER/WER is better, higher cosine similarity is better)
df_overall = df_results.groupby('model').agg({
    'cer': 'mean',
    'wer': 'mean',
    'cosine_similarity': 'mean',
    'sample_id': 'count'
}).round(4)

df_overall = df_overall.rename(columns={'sample_id': 'n_samples'})
df_overall = df_overall.sort_values('cer')

print("\nOverall Ranking (sorted by CER):")
display(df_overall)

## 4. Overall View - Visualizations

### Bar Charts - Average Metrics by Model

In [ ]:
# Create bar charts for each metric
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# CER
df_pivot_cer = df_results.groupby(['phase', 'model'])['cer'].mean().reset_index()
df_pivot_cer_wide = df_pivot_cer.pivot(index='model', columns='phase', values='cer')
df_pivot_cer_wide.plot(kind='bar', ax=axes[0], color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[0].set_title('Character Error Rate (CER) by Model', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Model', fontsize=10)
axes[0].set_ylabel('CER (lower is better)', fontsize=10)
axes[0].legend(title='Phase', loc='upper right')
axes[0].grid(axis='y', alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# WER
df_pivot_wer = df_results.groupby(['phase', 'model'])['wer'].mean().reset_index()
df_pivot_wer_wide = df_pivot_wer.pivot(index='model', columns='phase', values='wer')
df_pivot_wer_wide.plot(kind='bar', ax=axes[1], color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[1].set_title('Word Error Rate (WER) by Model', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Model', fontsize=10)
axes[1].set_ylabel('WER (lower is better)', fontsize=10)
axes[1].legend(title='Phase', loc='upper right')
axes[1].grid(axis='y', alpha=0.3)
axes[1].tick_params(axis='x', rotation=45)

# Cosine Similarity
df_pivot_cos = df_results.groupby(['phase', 'model'])['cosine_similarity'].mean().reset_index()
df_pivot_cos_wide = df_pivot_cos.pivot(index='model', columns='phase', values='cosine_similarity')
df_pivot_cos_wide.plot(kind='bar', ax=axes[2], color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[2].set_title('Cosine Similarity by Model', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Model', fontsize=10)
axes[2].set_ylabel('Cosine Similarity (higher is better)', fontsize=10)
axes[2].legend(title='Phase', loc='lower right')
axes[2].grid(axis='y', alpha=0.3)
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### Box Plots - Distribution of Metrics

In [ ]:
# Box plots for metric distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# CER distribution
sns.boxplot(data=df_results, x='model', y='cer', hue='phase', ax=axes[0])
axes[0].set_title('CER Distribution by Model', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Model', fontsize=10)
axes[0].set_ylabel('CER', fontsize=10)
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(title='Phase', loc='upper right')
axes[0].grid(axis='y', alpha=0.3)

# WER distribution
sns.boxplot(data=df_results, x='model', y='wer', hue='phase', ax=axes[1])
axes[1].set_title('WER Distribution by Model', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Model', fontsize=10)
axes[1].set_ylabel('WER', fontsize=10)
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Phase', loc='upper right')
axes[1].grid(axis='y', alpha=0.3)

# Cosine Similarity distribution
sns.boxplot(data=df_results, x='model', y='cosine_similarity', hue='phase', ax=axes[2])
axes[2].set_title('Cosine Similarity Distribution by Model', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Model', fontsize=10)
axes[2].set_ylabel('Cosine Similarity', fontsize=10)
axes[2].tick_params(axis='x', rotation=45)
axes[2].legend(title='Phase', loc='lower right')
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Box plot - Ground Truth in Prediction
plt.figure(figsize=(12, 6))
box_data = [results_df[results_df['model'] == model]['ground_truth_in_prediction'].values 
            for model in models]
bp = plt.boxplot(box_data, labels=models, patch_artist=True)
for patch, model in zip(bp['boxes'], models):
    patch.set_facecolor(get_model_color(model))
plt.xlabel('Model')
plt.ylabel('Ground Truth in Prediction')
plt.title('Ground Truth in Prediction Distribution by Model')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


### Heatmap - Model Performance Comparison

In [ ]:
# Create heatmap showing all metrics for all models
df_heatmap = df_results.groupby('model').agg({
    'cer': 'mean',
    'wer': 'mean',
    'cosine_similarity': 'mean'
}).round(4)

# Normalize metrics (0-1 scale, where 1 is best)
# For CER/WER: invert so lower is better -> higher score
df_heatmap_norm = df_heatmap.copy()
df_heatmap_norm['cer'] = 1 - (df_heatmap_norm['cer'] / df_heatmap_norm['cer'].max())
df_heatmap_norm['wer'] = 1 - (df_heatmap_norm['wer'] / df_heatmap_norm['wer'].max())
# Cosine similarity already higher is better

# Rename columns for clarity
df_heatmap_norm.columns = ['CER (normalized)', 'WER (normalized)', 'Cosine Similarity']

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df_heatmap_norm, annot=True, fmt='.3f', cmap='RdYlGn', 
            cbar_kws={'label': 'Performance (higher is better)'}, ax=ax,
            linewidths=0.5, linecolor='gray')
ax.set_title('Model Performance Heatmap (Normalized Metrics)', fontsize=14, fontweight='bold')
ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Model', fontsize=12)
plt.tight_layout()
plt.show()

# Also show raw values
print("\nRaw Metric Values:")
display(df_heatmap)

### Language-Specific Performance Analysis

### Scatter Plot: CER vs WER Correlation

### Error Analysis: Best and Worst Samples

### Inference Time Analysis

In [ ]:
# Analyze inference times across models and phases
inference_data = []

for phase, df in phase_dfs.items():
    time_cols = [col for col in df.columns if col.startswith('inference_time_ms_')]
    
    for time_col in time_cols:
        model = time_col.replace('inference_time_ms_', '')
        
        if df[time_col].notna().any():
            mean_time = df[time_col].mean()
            median_time = df[time_col].median()
            
            inference_data.append({
                'Phase': phase,
                'Model': model,
                'Mean Time (ms)': mean_time,
                'Median Time (ms)': median_time
            })

if inference_data:
    inference_df = pd.DataFrame(inference_data)
    
    print("\nInference Time Summary:")
    print("="*100)
    display(inference_df.sort_values(['Phase', 'Mean Time (ms)']))
    
    # Visualization
    if len(inference_df) > 0:
        fig, ax = plt.subplots(figsize=(12, 6))
        
        pivot_time = inference_df.pivot_table(index='Model', columns='Phase', values='Mean Time (ms)', fill_value=0)
        if not pivot_time.empty:
            colors = ['#e74c3c', '#3498db', '#2ecc71'][:len(pivot_time.columns)]
            pivot_time.plot(kind='bar', ax=ax, color=colors, alpha=0.8)
            
            ax.set_title('Average Inference Time by Model and Phase', fontsize=14, fontweight='bold')
            ax.set_xlabel('Model', fontsize=12)
            ax.set_ylabel('Time (ms)', fontsize=12)
            ax.legend(title='Phase', title_fontsize=11, fontsize=10)
            ax.grid(axis='y', alpha=0.3)
            ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
            
            plt.tight_layout()
            plt.show()
else:
    print("No inference time data available")

In [ ]:
# Analyze best and worst performing samples for Phase A
if 'Pa' in phase_dfs:
    df_analysis = df_results[df_results['phase'] == 'Pa'].copy()
    
    # Get first model for analysis
    first_model = df_analysis['model'].iloc[0] if len(df_analysis) > 0 else None
    
    if first_model:
        # Get best and worst samples
        best_samples = df_analysis.nsmallest(5, 'cer')
        worst_samples = df_analysis.nlargest(5, 'cer')
        
        print("\n" + "="*120)
        print(f"BEST PERFORMING SAMPLES (Lowest CER) - Model: {first_model}")
        print("="*120)
        
        for _, row in best_samples.iterrows():
            sample_id = row['sample_id']
            sample_row = df_pa.iloc[int(sample_id)] if int(sample_id) < len(df_pa) else None
            
            if sample_row is not None:
                print(f"\nSample: {sample_id} | CER: {row['cer']:.4f} | Language: {row['language']}")
                print(f"Ground Truth: {sample_row['ground_truth'][:100]}..." if len(str(sample_row['ground_truth'])) > 100 else f"Ground Truth: {sample_row['ground_truth']}")
                
                pred_col = f'prediction_{first_model}'
                if pred_col in sample_row and sample_row[pred_col]:
                    pred_text = str(sample_row[pred_col])
                    print(f"Prediction:   {pred_text[:100]}..." if len(pred_text) > 100 else f"Prediction:   {pred_text}")
        
        print("\n" + "="*120)
        print(f"WORST PERFORMING SAMPLES (Highest CER) - Model: {first_model}")
        print("="*120)
        
        for _, row in worst_samples.iterrows():
            sample_id = row['sample_id']
            sample_row = df_pa.iloc[int(sample_id)] if int(sample_id) < len(df_pa) else None
            
            if sample_row is not None:
                print(f"\nSample: {sample_id} | CER: {row['cer']:.4f} | Language: {row['language']}")
                print(f"Ground Truth: {sample_row['ground_truth'][:100]}..." if len(str(sample_row['ground_truth'])) > 100 else f"Ground Truth: {sample_row['ground_truth']}")
                
                pred_col = f'prediction_{first_model}'
                if pred_col in sample_row and sample_row[pred_col]:
                    pred_text = str(sample_row[pred_col])
                    print(f"Prediction:   {pred_text[:100]}..." if len(pred_text) > 100 else f"Prediction:   {pred_text}")

In [ ]:
# Create summary for scatter plot
summary_data = []
for phase, df in phase_dfs.items():
    phase_results = df_results[df_results['phase'] == phase]
    for model in phase_results['model'].unique():
        model_data = phase_results[phase_results['model'] == model]
        summary_data.append({
            'Phase': phase,
            'Model': model,
            'CER': model_data['cer'].mean(),
            'WER': model_data['wer'].mean(),
            'Cosine Similarity': model_data['cosine_similarity'].mean(),
        })

summary_df = pd.DataFrame(summary_data)

# Scatter plot of CER vs WER for each phase
phases = sorted(summary_df['Phase'].unique())
fig, axes = plt.subplots(1, len(phases), figsize=(6 * len(phases), 6))

if len(phases) == 1:
    axes = [axes]

for idx, phase in enumerate(phases):
    ax = axes[idx]
    
    phase_data = summary_df[summary_df['Phase'] == phase]
    
    scatter = ax.scatter(phase_data['CER'], phase_data['WER'], 
                        s=200, alpha=0.6, c=range(len(phase_data)), cmap='viridis')
    
    # Add model labels
    for _, row in phase_data.iterrows():
        ax.annotate(row['Model'], (row['CER'], row['WER']), 
                   fontsize=9, ha='right', va='bottom')
    
    ax.set_title(f'Phase {phase} - CER vs WER', fontsize=14, fontweight='bold')
    ax.set_xlabel('CER (Character Error Rate)', fontsize=12)
    ax.set_ylabel('WER (Word Error Rate)', fontsize=12)
    ax.grid(alpha=0.3)
    
    # Add diagonal reference line
    lims = [0, max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lims, lims, 'r--', alpha=0.3, label='CER=WER')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Aggregate metrics by language and model
df_lang_perf = df_results.groupby(['language', 'model']).agg({
    'cer': 'mean',
    'wer': 'mean',
    'cosine_similarity': 'mean',
    'sample_id': 'count'
}).round(4)

df_lang_perf = df_lang_perf.rename(columns={'sample_id': 'n_samples'})
df_lang_perf = df_lang_perf.reset_index()

print("Performance by Language and Model:")
display(df_lang_perf.sort_values(['language', 'cer']))

# Best model per language (by CER)
best_per_language = df_lang_perf.loc[df_lang_perf.groupby('language')['cer'].idxmin()]
print("\nBest Model per Language (by CER):")
display(best_per_language[['language', 'model', 'cer', 'wer', 'cosine_similarity']])

### Heatmap - CER by Language and Model

In [ ]:
# Create heatmap of CER by language and model
df_lang_pivot = df_results.groupby(['language', 'model'])['cer'].mean().reset_index()
df_lang_pivot_wide = df_lang_pivot.pivot(index='language', columns='model', values='cer')

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(df_lang_pivot_wide, annot=True, fmt='.3f', cmap='RdYlGn_r', 
            cbar_kws={'label': 'CER (lower is better)'}, ax=ax,
            linewidths=0.5, linecolor='gray')
ax.set_title('Character Error Rate by Language and Model', fontsize=14, fontweight='bold')
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Language', fontsize=12)
plt.tight_layout()
plt.show()

### Bar Charts - Performance by Language

In [ ]:
# Show CER for each language across all models
languages = df_results['language'].unique()
n_languages = len(languages)
n_cols = 3
n_rows = (n_languages + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten() if n_languages > 1 else [axes]

for idx, language in enumerate(sorted(languages)):
    df_lang = df_results[df_results['language'] == language]
    df_lang_agg = df_lang.groupby('model')['cer'].mean().sort_values()
    
    axes[idx].bar(range(len(df_lang_agg)), df_lang_agg.values, color='steelblue')
    axes[idx].set_xticks(range(len(df_lang_agg)))
    axes[idx].set_xticklabels(df_lang_agg.index, rotation=45, ha='right')
    axes[idx].set_title(f'{language}', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('CER', fontsize=10)
    axes[idx].grid(axis='y', alpha=0.3)

# Hide unused subplots
for idx in range(n_languages, len(axes)):
    axes[idx].axis('off')

plt.suptitle('CER Performance by Language (sorted by model performance)', 
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

### Language Difficulty Ranking

In [ ]:
# Average CER across all models per language (language difficulty)
df_lang_difficulty = df_results.groupby('language').agg({
    'cer': ['mean', 'std'],
    'wer': ['mean', 'std'],
    'ground_truth_in_prediction': ['mean', 'sum'],
    'cosine_similarity': ['mean', 'std'],
    'sample_id': 'count'
}).round(4)

df_lang_difficulty.columns = ['_'.join(col).strip() for col in df_lang_difficulty.columns.values]
df_lang_difficulty = df_lang_difficulty.rename(columns={'sample_id_count': 'n_samples'})
df_lang_difficulty = df_lang_difficulty.sort_values('cer_mean', ascending=False)

print("Language Difficulty Ranking (by average CER across all models):")
display(df_lang_difficulty)

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
df_lang_difficulty['cer_mean'].plot(kind='barh', ax=ax, color='coral')
ax.set_title('Average CER by Language (higher = more difficult)', fontsize=14, fontweight='bold')
ax.set_xlabel('Average CER', fontsize=12)
ax.set_ylabel('Language', fontsize=12)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

### Phase Comparison - Impact of Context

In [ ]:
# Compare VLM performance: Pb (baseline) vs Pc (with context)
vlm_models = list(set(models_pb).intersection(models_pc))

if vlm_models:
    df_vlm_comparison = df_results[df_results['model'].isin(vlm_models)].copy()
    df_vlm_comparison = df_vlm_comparison[df_vlm_comparison['phase'].isin(['Pb', 'Pc'])]
    
    df_vlm_agg = df_vlm_comparison.groupby(['phase', 'model']).agg({
        'cer': 'mean',
        'wer': 'mean',
        'cosine_similarity': 'mean'
    }).round(4).reset_index()
    
    print("VLM Phase Comparison (Pb: baseline vs Pc: with context):")
    display(df_vlm_agg)
    
    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for idx, metric in enumerate(['cer', 'wer', 'cosine_similarity']):
        df_pivot = df_vlm_agg.pivot(index='model', columns='phase', values=metric)
        df_pivot.plot(kind='bar', ax=axes[idx], color=['#ff7f0e', '#2ca02c'])
        axes[idx].set_title(f'{metric.upper()} - Phase Comparison', fontsize=12, fontweight='bold')
        axes[idx].set_xlabel('Model', fontsize=10)
        axes[idx].set_ylabel(metric.upper(), fontsize=10)
        axes[idx].legend(title='Phase', loc='best')
        axes[idx].grid(axis='y', alpha=0.3)
        axes[idx].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
else:
    print("No common VLM models found in both Pb and Pc phases")

## 7. LLM Query Section

## 6. Save Embeddings Cache

In [ ]:
# Save any remaining embeddings to disk for future use
print("\n📁 Finalizing embedding cache...")

if embedding_manager.modified_phases:
    print(f"Saving embeddings for modified phases: {embedding_manager.modified_phases}")
    
    for phase in embedding_manager.modified_phases:
        if phase in embedding_manager.cache:
            saved_file = save_embeddings_for_phase(
                dataset_name=DATASET_NAME,
                phase=phase,
                embeddings_dict=embedding_manager.cache[phase],
                embeddings_base_dir=EMBEDDINGS_DIR
            )
            print(f"✅ Saved: {saved_file.name}")
    
    print("\n✅ All embeddings have been persisted to disk!")
else:
    print("✅ No new embeddings to save (all from cache)")

### Placeholder for LLM Analysis

This section is reserved for querying an LLM to analyze the notebook outputs.

**Instructions for LLM:**
1. Review all visualizations and statistical outputs above
2. Identify key findings:
   - Best performing model overall
   - Best OCR vs VLM comparison
   - Impact of context (Phase B vs Phase C)
   - Language-specific performance patterns
   - Most/least difficult languages
3. Provide insights on:
   - Which models excel at which languages
   - Whether VLMs benefit from task-specific context
   - Trade-offs between accuracy metrics (CER/WER vs semantic similarity)
4. Recommend:
   - Best model for multilingual OCR
   - When to use OCR vs VLM
   - Language-specific model selection strategies

In [ ]:
# Placeholder for LLM query code
# This cell can be populated with code to:
# 1. Export key statistics and visualizations
# 2. Generate prompts for LLM analysis
# 3. Display LLM-generated insights

print("LLM analysis section - ready for implementation")

### Summary Statistics Export

In [ ]:
# Export key statistics for LLM analysis
summary = {
    'dataset': 'ICDAR_mini',
    'n_samples': len(df_pa),
    'n_languages': df_results['language'].nunique(),
    'languages': sorted(df_results['language'].unique().tolist()),
    'models': {
        'ocr': models_pa,
        'vlm_baseline': models_pb,
        'vlm_context': models_pc
    },
    'best_overall': df_overall.index[0],
    'best_cer': df_overall['cer'].min(),
    'metrics': df_agg.to_dict('records')
}

print("Summary Statistics:")
print(json.dumps(summary, indent=2))

# Optionally save to file
# with open('icdar_summary.json', 'w') as f:
#     json.dump(summary, f, indent=2)

## 8. Conclusion

### Key Deliverables from This Analysis

This notebook has generated comprehensive benchmark analysis for the ICDAR multilingual text recognition dataset including:

1. **Metrics Evaluation**
   - Character Error Rate (CER) for all models across all languages
   - Word Error Rate (WER) for word-level accuracy assessment
   - Cosine Similarity for semantic similarity analysis
   - Ground Truth in Prediction metric for text presence analysis

2. **Performance Visualizations**
   - Bar charts showing average metrics by model and phase
   - Box plots showing metric distributions
   - Heatmaps showing model performance comparison
   - Language-specific performance heatmaps
   - CER vs WER correlation scatter plots
   - Inference time analysis by model

3. **Error Analysis**
   - Best performing samples (lowest CER) for understanding model strengths
   - Worst performing samples (highest CER) for identifying failure cases
   - Sample-level analysis by language

4. **Efficiency Analysis**
   - Inference time benchmarks per model and phase
   - Speed vs accuracy tradeoff analysis
   - Model efficiency rankings

5. **Cached Embeddings**
   - Pre-computed and saved embeddings for efficient cosine similarity calculations
   - Incremental saving during analysis to prevent recomputation

### Next Steps

1. **Model Selection**: Use CER/WER metrics to select best-performing models for deployment
2. **Language-Specific Strategy**: Consider language-specific model selection based on performance heatmaps
3. **Prompting Optimization**: Analyze Pb vs Pc phase differences to assess impact of task-aware prompting
4. **Error Pattern Analysis**: Deep dive into worst-performing samples to identify systematic errors
5. **Production Evaluation**: Use inference time data to plan deployment strategies balancing accuracy and latency